In [1]:
!pip install pyspark -q

In [2]:
import os

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

In [3]:
from google.colab import files
import shutil

uploaded = files.upload()   # a file picker appears, choose your sales CSV

file_name = list(uploaded.keys())[0]
shutil.move(file_name, "data_lake/bronze/sales.csv")

Saving sales (3).csv to sales (3) (1).csv


'data_lake/bronze/sales.csv'

In [4]:
for folder, subfolders, filenames in os.walk("data_lake"):
    print(folder, filenames)

data_lake []
data_lake/bronze ['sales.csv']
data_lake/silver []
data_lake/silver/preprocessed_sales []


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesDataLake").getOrCreate()

In [6]:
df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=True)

In [7]:
df.printSchema()
df.show(5)
print("Total records:", df.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.ansi.enabled", "false")

In [9]:
total = df.count()
print("Exact duplicate rows:", total - df.dropDuplicates().count())
print("Repeated order_ids:", total - df.select("order_id").distinct().count())

df.select([F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
           for c in df.columns]).show()

Exact duplicate rows: 10
Repeated order_ids: 18
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [10]:
text_cols = ["customer_name", "category", "city", "state", "payment_method", "order_status"]

for c in text_cols:
    print(c, "| extra spaces:", df.filter(F.col(c) != F.trim(F.col(c))).count())

for c in ["customer_name", "category", "city", "state"]:
    print(c, "| odd capitalization:", df.filter(F.trim(F.col(c)) != F.initcap(F.trim(F.col(c)))).count())

customer_name | extra spaces: 3
category | extra spaces: 0
city | extra spaces: 3
state | extra spaces: 0
payment_method | extra spaces: 0
order_status | extra spaces: 0
customer_name | odd capitalization: 0
category | odd capitalization: 6
city | odd capitalization: 6
state | odd capitalization: 0


In [11]:
print("quantity <= 0:", df.filter(F.col("quantity") <= 0).count())
print("unit_price <= 0:", df.filter(F.col("unit_price") <= 0).count())
print("discount outside 0-100:", df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count())

quantity <= 0: 12
unit_price <= 0: 8
discount outside 0-100: 8


In [12]:
formats = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "MM-dd-yyyy"]
parsed = F.coalesce(*[F.to_date("order_date", f) for f in formats])

print("Not in yyyy-MM-dd format:", df.filter(~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")).count())
print("Cannot be parsed at all:", df.filter(parsed.isNull() & F.col("order_date").isNotNull()).count())
df.filter(parsed.isNull()).select("order_date").show()

Not in yyyy-MM-dd format: 4
Cannot be parsed at all: 4
+----------+
|order_date|
+----------+
|31/02/2026|
|not-a-date|
|2025-02-30|
|2026-13-05|
+----------+



In [13]:
silver = df.dropDuplicates()

for c in text_cols:
    silver = silver.withColumn(c, F.trim(F.col(c)))
    silver = silver.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

for c in ["customer_name", "category", "city", "state"]:
    silver = silver.withColumn(c, F.initcap(c))

silver = silver.fillna("Unknown", subset=["customer_name", "payment_method", "city"])

silver = (silver
    .withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
    .withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
    .withColumn("discount_percent", F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent"))))

median_qty = int(silver.approxQuantile("quantity", [0.5], 0.01)[0])
by_product = Window.partitionBy("product")

silver = silver.withColumn("unit_price", F.coalesce("unit_price", F.percentile_approx("unit_price", 0.5).over(by_product)))
silver = silver.fillna({"quantity": median_qty, "discount_percent": 0})

silver = silver.withColumn("order_date", parsed).orderBy("order_id")

In [14]:
import glob, shutil

silver.coalesce(1).write.mode("overwrite").option("header", True).csv("data_lake/silver/preprocessed_sales")

part_file = glob.glob("data_lake/silver/preprocessed_sales/part-*.csv")[0]
shutil.copy(part_file, "data_lake/silver/sales_silver.csv")

'data_lake/silver/sales_silver.csv'

In [15]:
print("Rows before:", total, "| Rows after:", silver.count())

silver_check = spark.read.csv("data_lake/silver/sales_silver.csv", header=True, inferSchema=True)
silver_check.show(5)
silver_check.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_check.columns]).show()

Rows before: 1000 | Rows after: 990
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|       680|               5|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|      3080|              12|           UPI|      Pune|   Maharashtra|2025-05-06|

In [16]:
from google.colab import files
files.download("data_lake/silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>